In [1]:
import ccxt
import pandas as pd
import time 

In [2]:
# Binanceのインスタンスを作成（レートリミットを有効化）
exchange = ccxt.binance({
    'enableRateLimit': True,
})

# BTC/USDTのデータを取得
symbol = 'BTC/USDT'
timeframe = '1h'  # 15分足のデータ
since = exchange.parse8601('2020-01-01T00:00:00Z')  # データの開始日

# データを取得する関数（例外処理と進行状況の追加）
def fetch_all_ohlcv(symbol, timeframe, since):
    all_ohlcv = []
    now = exchange.milliseconds()
    while since < now:
        try:
            ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since, limit=1000)
            if not ohlcv:
                break
            since = ohlcv[-1][0] + 1  # タイムスタンプを更新
            all_ohlcv.extend(ohlcv)
        except Exception as e:
            print(f'An error occurred: {e}')
            # リトライのために一時停止
            time.sleep(5)
            continue
    return all_ohlcv

# データを取得
ohlcv = fetch_all_ohlcv(symbol, timeframe, since)
data = pd.DataFrame(ohlcv, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])

# タイムスタンプを変換
data['timestamp'] = pd.to_datetime(data['timestamp'], unit='ms')

# CSVファイルとして保存
data.to_csv('btc_usdt_1h_2018_to_now.csv', index=False)

print('Data fetching and saving completed.')

Data fetching and saving completed.


In [3]:
# データの読み込みと文字列として保存されている日時情報を日時型のデータに変換
df = pd.read_csv('btc_usdt_1h_2018_to_now.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [4]:
display(df.head())
display(df.tail())

,timestamp,open,high,low,close,volume
0,2020-01-01 00:00:00,7195.24,7196.25,7175.46,7177.02,511.814901
1,2020-01-01 01:00:00,7176.47,7230.00,7175.71,7216.27,883.052603
2,2020-01-01 02:00:00,7215.52,7244.87,7211.41,7242.85,655.156809
3,2020-01-01 03:00:00,7242.66,7245.00,7220.00,7225.01,783.724867
4,2020-01-01 04:00:00,7225.00,7230.00,7215.03,7217.27,467.812578


,timestamp,open,high,low,close,volume
46514,2025-04-23 10:00:00,93888.53,93900.00,93340.81,93745.19,1429.75497
46515,2025-04-23 11:00:00,93745.04,93777.95,93324.45,93466.01,713.03617
46516,2025-04-23 12:00:00,93466.01,93857.14,93421.94,93731.40,715.78459
46517,2025-04-23 13:00:00,93731.40,94696.05,92872.16,93020.22,3287.20024
46518,2025-04-23 14:00:00,93020.35,93665.99,92666.01,93189.51,1171.40938


In [5]:
print(df.head())

            timestamp     open     high      low    close      volume
0 2020-01-01 00:00:00  7195.24  7196.25  7175.46  7177.02  511.814901
1 2020-01-01 01:00:00  7176.47  7230.00  7175.71  7216.27  883.052603
2 2020-01-01 02:00:00  7215.52  7244.87  7211.41  7242.85  655.156809
3 2020-01-01 03:00:00  7242.66  7245.00  7220.00  7225.01  783.724867
4 2020-01-01 04:00:00  7225.00  7230.00  7215.03  7217.27  467.812578


In [6]:
import pandas as pd

# ファイルパス
etf_data_path = 'BTC_ETF.csv'  # ETFデータのファイル
ohlcv_data_path = 'btc_usdt_1h_2018_to_now.csv'  # OHLCVデータのファイル
output_path = 'merged_btc_ohlcv_etf.csv'  # 結合後のCSV出力先

# CSVを読み込む
etf_data = pd.read_csv(etf_data_path)
ohlcv_data = pd.read_csv(ohlcv_data_path)

# ETFデータの日付をdatetimeに変換
etf_data['Date'] = pd.to_datetime(etf_data['Date'], errors='coerce')

# OHLCVデータのタイムスタンプをdatetimeに変換
ohlcv_data['timestamp'] = pd.to_datetime(ohlcv_data['timestamp'], errors='coerce')

# OHLCVデータに日付列を追加
ohlcv_data['date'] = ohlcv_data['timestamp'].dt.date

# ETFデータの日付と一致させるために日付型に変換
ohlcv_data['date'] = pd.to_datetime(ohlcv_data['date'])

# 結合を実施（left joinでOHLCVデータを基準にする）
merged_data = pd.merge(
    ohlcv_data,
    etf_data.rename(columns={'Date': 'date'}),  # 'Date'列を'ohlcv_data'の'date'列に合わせる
    how='left',
    on='date'
)

# カラムの順番を調整（元のOHLCVデータ列を先に）
columns_to_keep = list(ohlcv_data.columns) + list(etf_data.columns[1:])  # ETFの時間列は含まない
merged_data = merged_data[columns_to_keep]
merged_data = merged_data.drop(columns=['date'], errors='ignore')

# CSVに保存
merged_data.to_csv(output_path, index=False)



print(f"結合されたデータは {output_path} に保存されました。")

結合されたデータは merged_btc_ohlcv_etf.csv に保存されました。
